## 1. Environment Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import ast
import re

# === PLOT SETTINGS ===
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

# === 1. DATA LOADING AND PREPROCESSING ===
print("📥 Loading dataset...")
df = pd.read_csv("/dataset/semantic_benchmark_with_emb.csv")

# Safe embedding parsing
def parse_embedding(val):
    if isinstance(val, str):
        # If embedding is stored as a string "[0.1, 0.2...]"
        return np.array(ast.literal_eval(val), dtype=np.float32)
    return val

df['clap_emb'] = df['clap_emb'].apply(parse_embedding)

df['asm_len'] = df['raw_asm'].astype(str).apply(lambda x: len(x.split('\n')))
df = df[df['asm_len'] >= 10].reset_index(drop=True)

# df['clap_hash'] = df['clap_emb'].apply(lambda v: str(v[:30]))
# df = df.drop_duplicates(subset=['clap_hash'], inplace=False).reset_index(drop=True)

asm_vectors = np.stack(df['clap_emb'].values)
asm_tensors = torch.tensor(asm_vectors)

# Normalize vectors for cosine similarity
asm_tensors = F.normalize(asm_tensors, p=2, dim=1)

print(f"✅ Dataset ready: {asm_tensors.shape[0]} vectors, dimension {asm_tensors.shape[1]}")

## 2. Exploratory Data Analysis (EDA) & Vector Sanity Check

In [ ]:
# 1. Data size check
print(f"📊 Total rows in DataFrame: {len(df)}")

# 2. Metadata parsing check
print("\n🔍 Sample parsed data (first 5 rows):")
print(df[['category', 'behavior', 'obfuscation', 'compiler']].head(5))

print("\n📈 Category distribution:")
print(df['category'].value_counts())

print("\n📈 Obfuscation distribution:")
print(df['obfuscation'].value_counts())

# 3. VECTOR DUPLICATE CHECK (Most important!)
# Convert tensors to numpy for analysis
vecs = asm_tensors.numpy()
unique_vecs = np.unique(vecs, axis=0)  # Round for robustness
n_unique = unique_vecs.shape[0]

print(f"\n👯 Unique vectors: {n_unique} out of {len(df)}")
print(f"   Duplicate rate: {100 * (1 - n_unique/len(df)):.2f}%")

if n_unique < 50:
    print("🚨 CRITICAL ISSUE: Model produces identical vectors for different functions!")
    print("   Possible causes: clean_and_rebase removes too much information, or functions are too short.")
else:
    print("✅ Vectors are diverse. The issue is likely in t-SNE parameters.")

# 4. ASM length / sanity check
df['asm_len'] = df['clap_emb'].apply(lambda x: len(str(x)))  # Rough estimate
print("\n📝 Sample ASM (sanity check for emptiness):")

# Print raw ASM sample if available
if 'raw_asm' in df.columns:
    print(df['raw_asm'].iloc[0][:300])

## 3. Visualizing the Semantic Space (t-SNE)

In [ ]:
# === 2. SPACE VISUALIZATION (t-SNE) ===
print("🗺️ Building t-SNE map...")

tsne = TSNE(n_components=2, perplexity=20, random_state=42)
embeddings_2d = tsne.fit_transform(asm_tensors.numpy())

df['tsne_x'] = embeddings_2d[:, 0]
df['tsne_y'] = embeddings_2d[:, 1]

plt.figure(figsize=(14, 10))
sns.scatterplot(
    data=df,
    x='tsne_x',
    y='tsne_y',
    hue='category',
    style='obfuscation',
    palette='tab10',
    s=100,
    alpha=0.8
)

plt.title("Semantic Space of Compiled Functions (t-SNE)", fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()

plt.savefig("tsne_semantic_space.png")
plt.show()

In [ ]:
## 4. Model Initialization (CLAP-Text)

In [ ]:
# === 3. TEXT MODEL LOADING ===
print("🤖 Loading CLAP-Text model...")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEXT_MODEL = "hustwc/clap-text"

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, trust_remote_code=True)
text_model = AutoModel.from_pretrained(TEXT_MODEL, trust_remote_code=True).to(DEVICE).eval()

print("Done!")

In [ ]:
def encode_text_queries(queries):
    inputs = text_tokenizer(
        queries,
        padding=True,
        truncation=True,
        return_tensors='pt'
    ).to(DEVICE)

    with torch.no_grad():
        out = text_model(**inputs)

        if len(out.shape) == 3:
            out = out.mean(dim=1)

        emb = F.normalize(out, p=2, dim=0)

    return emb.cpu()

## Experiments

### Experiment 1: Cross-Modal Retrieval (Text-to-ASM)

In [ ]:
# === EXPERIMENT 1: CROSS-MODAL RETRIEVAL (Text -> ASM) ===
print("\n" + "="*50)
print("🎯 EXPERIMENT 1: Text-to-ASM Semantic Search")
print("="*50)

unique_behaviors = df['behavior'].unique()

# Generate natural language queries from behavior column
# (e.g. "RC4_Encrypt" -> "perform RC4 Encrypt")
query_texts = [beh.replace('_', ' ') for beh in unique_behaviors]
query_tensors = encode_text_queries(query_texts)

# Similarity matrix [Queries x All functions]
sim_matrix_cross = torch.matmul(query_tensors, asm_tensors.T)

retrieval_results = []

for i, behavior in enumerate(unique_behaviors):
    sims = sim_matrix_cross[i].numpy()
    sorted_indices = np.argsort(sims)[::-1]

    # Find any implementation of this behavior
    correct_mask = (df['behavior'].values[sorted_indices] == behavior)
    first_correct_rank = np.argmax(correct_mask) + 1

    retrieval_results.append({
        'behavior': behavior,
        'category': df[df['behavior'] == behavior]['category'].iloc[0],
        'first_rank': first_correct_rank,
        'r_at_1': 1 if first_correct_rank == 1 else 0,
        'r_at_5': 1 if first_correct_rank <= 5 else 0,
        'r_at_10': 1 if first_correct_rank <= 10 else 0,
        'mrr': 1.0 / first_correct_rank
    })

res_df = pd.DataFrame(retrieval_results)

print(f"Recall@1:  {res_df['r_at_1'].mean():.2%}")
print(f"Recall@5:  {res_df['r_at_5'].mean():.2%}")
print(f"Mean Reciprocal Rank (MRR): {res_df['mrr'].mean():.4f}")

# Metrics by category
cat_metrics = res_df.groupby('category')[['r_at_1', 'r_at_5']].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=cat_metrics.melt(id_vars='category'),
    y='category',
    x='value',
    hue='variable',
    palette='viridis'
)

plt.title("Text-to-ASM Retrieval Performance by Category")
plt.xlabel("Accuracy")
plt.ylabel("")
plt.tight_layout()

plt.savefig("retrieval_by_category.png")
plt.show()

#### 1.1.RELAXED EVALUATION (SUPER-INTENTS)

In [ ]:
# === EXPERIMENT 1.1: RELAXED EVALUATION (SUPER-INTENTS) ===

# Create grouping dictionary (Macro-Behaviors)
BEHAVIOR_MAPPING = {
    # 1. HTTP Communication
    "HTTP_Get_Request": "HTTP_Communication",
    "HTTP_Post_Request": "HTTP_Communication",
    "Download_File_From_URL": "HTTP_Communication",

    # 2. File IO
    "Read_File_Content": "File_IO",
    "Write_File_Content": "File_IO",
    "Append_To_Log_File": "File_IO",
    "Drop_File_From_Buffer": "File_IO",

    # 3. System Execution
    "Create_Scheduled_Task_XML_Import": "Persistence_Exec",
    "Scheduled_Task_Create": "Persistence_Exec",
    "Service_Create": "Persistence_Exec",

    # 4. Anti-Debugging
    "IsDebuggerPresent_Check": "Anti_Debug",
    "IFEO_Debugger_Hijack": "Anti_Debug",
    "GetTickCount_Timing_Check": "Anti_Debug",

    # 5. Cryptography (kept or grouped by similarity)
    "AES_Block_Encrypt": "Symmetric_Crypto",
    "RC4_Encrypt": "Symmetric_Crypto",
    "TEA_Block_Encrypt": "Symmetric_Crypto"
}

# Map behavior to super-class, fallback to original if missing
def get_super_intent(behavior):
    return BEHAVIOR_MAPPING.get(behavior, behavior)

# Apply mapping to DataFrame
df['macro_behavior'] = df['behavior'].apply(get_super_intent)

print("\n" + "="*50)
print("🎯 EXPERIMENT 2.1: Semantic Search with Macro-Behaviors (Relaxed Evaluation)")
print("="*50)

unique_macros = df['macro_behavior'].unique()
print(f"Number of classes reduced from {len(unique_behaviors)} to {len(unique_macros)}.")

relaxed_results = []

for macro in unique_macros:
    # Query is generated from macro intent
    query_text = f"function that performs {macro.replace('_', ' ')}"
    q_vec = encode_text_queries([query_text])

    sims = torch.matmul(q_vec, asm_tensors.T).numpy()[0]
    sorted_indices = np.argsort(sims)[::-1]

    # Correct answer = any function in macro group
    correct_mask = (df['macro_behavior'].values[sorted_indices] == macro)
    first_correct_rank = np.argmax(correct_mask) + 1

    relaxed_results.append({
        'macro_behavior': macro,
        'r_at_1': 1 if first_correct_rank == 1 else 0,
        'r_at_5': 1 if first_correct_rank <= 5 else 0,
        'mrr': 1.0 / first_correct_rank
    })

rel_df = pd.DataFrame(relaxed_results)

print(f"Relaxed Recall@1: {rel_df['r_at_1'].mean():.2%}")
print(f"Relaxed Recall@5: {rel_df['r_at_5'].mean():.2%}")
print(f"Relaxed MRR:      {rel_df['mrr'].mean():.4f}")

### Experiment 2: Robustness Analysis

In [ ]:
# === EXPERIMENT 2: ROBUSTNESS ANALYSIS ===
print("\n" + "="*50)
print("🛡️ EXPERIMENT 2: Robustness to Obfuscation and Optimizations")
print("="*50)

robustness_data = []

for i, behavior in enumerate(unique_behaviors):
    sims = sim_matrix_cross[i].numpy()
    target_samples = df[df['behavior'] == behavior]

    for _, row in target_samples.iterrows():
        rank = (sims > sims[row.name]).sum() + 1

        robustness_data.append({
            'behavior': behavior,
            'obfuscation': row['obfuscation'],
            'compiler': row['compiler'],
            'is_top_5': 1 if rank <= 5 else 0
        })

rob_df = pd.DataFrame(robustness_data)

# Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=rob_df,
    x='obfuscation',
    y='is_top_5',
    errorbar=None,
    palette='Reds_r',
    ax=ax1,
    order=['V1', 'V2', 'V3', 'V4', 'V5']
)

ax1.set_title("Robustness to Obfuscation (Recall@5)")
ax1.set_xlabel("Obfuscation Level")
ax1.set_ylabel("Probability of being in Top-5")
ax1.set_ylim(0, 1.1)

sns.barplot(
    data=rob_df,
    x='compiler',
    y='is_top_5',
    errorbar=None,
    palette='Greens_r',
    ax=ax2
)

ax2.set_title("Robustness to Compiler Flags (Recall@5)")
ax2.set_xlabel("Compiler Optimization")
ax2.set_ylabel("")
ax2.set_ylim(0, 1.1)

plt.tight_layout()

plt.savefig("robustness_analysis.png")
plt.show()

### Experiment 3: Binary Code Similarity (ASM-to-ASM)

In [ ]:
# === 6. EXPERIMENT 3: CODE-TO-CODE SIMILARITY (ASM to ASM) ===
print("\n" + "="*50)
print("🧬 EXPERIMENT 3: Binary Code Similarity (ASM-to-ASM)")
print("="*50)

# Compute similarity matrix for all functions [528, 528]
sim_matrix_asm = torch.matmul(asm_tensors, asm_tensors.T).numpy()

# Zero out diagonal (to prevent self-matching)
np.fill_diagonal(sim_matrix_asm, 0)

bcsd_results = []

for i in range(len(df)):
    target_behavior = df.iloc[i]['behavior']

    sims = sim_matrix_asm[i]
    sorted_indices = np.argsort(sims)[::-1]

    # Find first function with the same behavior
    correct_mask = (df['behavior'].values[sorted_indices] == target_behavior)
    first_rank = np.argmax(correct_mask) + 1

    bcsd_results.append({
        'behavior': target_behavior,
        'r_at_1': 1 if first_rank == 1 else 0
    })

bcsd_df = pd.DataFrame(bcsd_results)
print(f"ASM-to-ASM Recall@1: {bcsd_df['r_at_1'].mean():.2%} (Ability to detect clones)")

# Confusion heatmap
# Averaging cosine similarity between behaviors
pivot_df = df.copy()
pivot_df['idx'] = range(len(pivot_df))

category_sims = np.zeros((len(unique_behaviors), len(unique_behaviors)))

for i, b1 in enumerate(unique_behaviors):
    idx1 = pivot_df[pivot_df['behavior'] == b1]['idx'].values

    for j, b2 in enumerate(unique_behaviors):
        idx2 = pivot_df[pivot_df['behavior'] == b2]['idx'].values

        # Mean similarity between all functions of B1 and B2
        sub_matrix = sim_matrix_asm[np.ix_(idx1, idx2)]
        category_sims[i, j] = np.mean(sub_matrix)

plt.figure(figsize=(14, 12))
sns.heatmap(
    category_sims,
    xticklabels=unique_behaviors,
    yticklabels=unique_behaviors,
    cmap='magma',
    annot=False
)

plt.title("Behavior Similarity Heatmap (ASM-to-ASM)")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)

plt.tight_layout()
plt.savefig("behavior_heatmap.png")
plt.show()

print("\n✅ Analysis completed successfully! All plots have been saved.")

### Experiment 4: Linguistic Robustness & Prompt Engineering

In [ ]:
print("\n" + "="*50)
print("🗣️ EXPERIMENT 4: Linguistic Robustness (Prompt Sensitivity)")
print("="*50)

# Dictionary: Category -> List of prompts (from direct to abstract)
PROMPT_VARIATIONS = {
    "Reverse_TCP_Connect":[
        "Create a socket and connect to a remote IP address to establish a reverse shell",  # Direct
        "Call WSAStartup, socket, and connect APIs",  # Technical
        "Establish a backdoor communication channel to the attacker's command and control server"  # Abstract (no explicit socket/connect terms)
    ],
    "RC4_Encrypt":[
        "Implement the RC4 stream cipher with key scheduling and pseudo-random generation",  # Direct
        "Initialize an array of 256 bytes, swap elements, and XOR the data buffer",  # Technical
        "Obfuscate the payload using a symmetric cryptographic stream algorithm"  # Abstract
    ],
    "IsDebuggerPresent_Check":[
        "Check if a debugger is attached to the current process",  # Direct
        "Read the Process Environment Block (PEB) to find the BeingDebugged flag",  # Technical
        "Perform anti-analysis and evasion checks to prevent reverse engineering"  # Abstract
    ]
}

prompt_results = []

for category, prompts in PROMPT_VARIATIONS.items():
    print(f"\n--- Testing Category: {category} ---")

    # Find correct indices for this category
    correct_indices = df.index[df['behavior'] == category].tolist()

    for prompt in prompts:
        # Encode query
        q_vec = encode_text_queries([prompt])

        # Compute similarity with full dataset
        sims = torch.matmul(q_vec, asm_tensors.T).numpy()[0]
        sorted_ranks = np.argsort(sims)[::-1]

        # Find first correct match
        correct_mask = np.isin(sorted_ranks, correct_indices)
        first_correct_rank = np.argmax(correct_mask) + 1

        print(f"Rank {first_correct_rank:3d} | Query: '{prompt}'")

        prompt_results.append({
            "Category": category,
            "Query_Type": "Direct"
            if prompts.index(prompt) == 0
            else ("Technical" if prompts.index(prompt) == 1 else "Abstract"),
            "Rank": first_correct_rank
        })

prompt_df = pd.DataFrame(prompt_results)

sns.catplot(
    data=prompt_df,
    x="Query_Type",
    y="Rank",
    hue="Category",
    kind="bar",
    height=5,
    aspect=1.5
)

plt.title("Effect of Prompt Formulation on Retrieval Rank (Lower is Better)")
plt.savefig("prompt_sensitivity.png")
plt.show()

### Experiment 5: "Needle in a Haystack" Simulation

In [ ]:
print("\n" + "="*50)
print("🪡 EXPERIMENT 5: Needle in a Haystack")
print("="*50)

# Build "haystack" (all benign functions)
benign_indices = df.index[df['category'] == 'Benign_Utility'].tolist()
haystack_tensors = asm_tensors[benign_indices]

# Test several "needles" (malicious behaviors)
needles_to_test = [
    "Reverse_TCP_Connect",
    "AES_Block_Encrypt",
    "GetTickCount_Timing_Check",
    "Check_Admin_Privileges",
    "Heap_Spray_Simulation",
    "Microphone_Record_Simulation",
    "IsDebuggerPresent_Check"
]

for needle in needles_to_test:
    # Take ONE implementation of the needle (e.g., V1 GCC_O2)
    needle_idx_list = df.index[
        (df['behavior'] == needle) &
        (df['compiler'] == 'gcc_O2') &
        (df['obfuscation'] == 'V2')
    ].tolist()

    if needle_idx_list:
        needle_idx = needle_idx_list[0]
    else:
        print(f"⚠️ Needle '{needle}' not found in dataset! Skipping.")
        continue

    needle_tensor = asm_tensors[needle_idx].unsqueeze(0)

    # Create mixed database: 1 malware + all benign functions
    mixed_db = torch.cat([needle_tensor, haystack_tensors], dim=0)

    # Build query
    query = f"function that implements {needle.replace('_', ' ')}"
    q_vec = encode_text_queries([query])

    # Search
    sims = torch.matmul(q_vec, mixed_db.T).numpy()[0]
    sorted_ranks = np.argsort(sims)[::-1]

    # Needle is at index 0 in mixed_db
    needle_rank = np.where(sorted_ranks == 0)[0][0] + 1

    print(f"🎯 Query: '{query}'")
    print(f"   Database size: 1 malware + {len(benign_indices)} benign functions")
    print(f"   Result: needle found at rank {needle_rank}!")

    if needle_rank == 1:
        print("   ✅ PERFECT! Model immediately isolates the malicious sample.")
    elif needle_rank <= 5:
        print("   ⚠️ GOOD! Analyst would inspect only top 5 results.")

    print("-" * 40)

### Experiment 6: Specific Query -> Macro Target

In [ ]:
print("\n" + "="*50)
print("🚀 EXPERIMENT 6: Specific Query -> Macro Target")
print("="*50)

# 1. Take original (fine-grained) behaviors
unique_behaviors = df['behavior'].unique()

aggressive_results = []

for behavior in unique_behaviors:
    # 1. Use SPECIFIC query (strongest signal)
    query_text = behavior.replace('_', ' ')
    q_vec = encode_text_queries([query_text])

    # 2. Define target macro-group
    target_macro = get_aggressive_intent(behavior)

    # 3. Retrieval
    sims = torch.matmul(q_vec, asm_tensors.T).numpy()[0]
    sorted_indices = np.argsort(sims)[::-1]

    # 4. Success condition: any function from the same MACRO group is found
    # (e.g., searching RC4 and retrieving AES -> success since both are Crypto_Logic)
    found_macros = df['macro_behavior'].values[sorted_indices]
    correct_mask = (found_macros == target_macro)

    first_rank = np.argmax(correct_mask) + 1

    aggressive_results.append({
        'behavior': behavior,
        'target_macro': target_macro,
        'r_at_1': 1 if first_rank == 1 else 0,
        'r_at_5': 1 if first_rank <= 5 else 0,
        'mrr': 1.0 / first_rank
    })

agg_df = pd.DataFrame(aggressive_results)

print(f"True Aggressive Recall@1: {agg_df['r_at_1'].mean():.2%}")
print(f"True Aggressive Recall@5: {agg_df['r_at_5'].mean():.2%}")
print(f"True Aggressive MRR:      {agg_df['mrr'].mean():.4f}")

# Compare improvement
original_r5 = res_df['r_at_5'].mean()
new_r5 = agg_df['r_at_5'].mean()

print(f"\n📈 Accuracy gain from macro-grouping: +{(new_r5 - original_r5)*100:.2f}%")

### Experiment 7: Breakdown by Obfuscation Level

In [ ]:
print("\n" + "="*50)
print("⚖️ EXPERIMENT 7: Breakdown by Obfuscation Level")
print("="*50)

versions = ['V1', 'V2', 'V3', 'V4', 'V5']
results_by_version = []

# Create queries from all behaviors
unique_behaviors = df['behavior'].unique()
q_texts = [b.replace('_', ' ') for b in unique_behaviors]
q_vecs = encode_text_queries(q_texts)

# Full similarity matrix [78 queries x 1000+ functions]
sim_matrix_full = torch.matmul(q_vecs, asm_tensors.T).numpy()

for v in versions:
    # Restrict search space to ONLY functions of version V
    # Simulates: "Attacker uses V3. Can we detect it?"
    
    target_indices = df.index[df['obfuscation'] == v].tolist()
    if not target_indices:
        continue

    # Subset similarity matrix [queries x V-specific functions]
    sim_sub = sim_matrix_full[:, target_indices]

    current_recalls = []

    for i, behavior in enumerate(unique_behaviors):
        # Which columns correspond to this behavior?
        # We compare within the same obfuscation level only

        row_sims = sim_sub[i]
        sorted_local_idxs = np.argsort(row_sims)[::-1]

        # Map local indices back to global indices
        sorted_global_idxs = [target_indices[k] for k in sorted_local_idxs]

        # Identify correct matches
        correct_mask = df.loc[sorted_global_idxs, 'behavior'] == behavior

        if correct_mask.sum() == 0:
            current_recalls.append(0)
        else:
            rank = np.argmax(correct_mask.values) + 1
            current_recalls.append(1 if rank <= 5 else 0)

    avg_recall = np.mean(current_recalls)
    results_by_version.append({'version': v, 'recall_at_5': avg_recall})

    print(f"Recall@5 on {v} only: {avg_recall:.2%}")

# Visualization of performance drop
ver_df = pd.DataFrame(results_by_version)

plt.figure(figsize=(8, 5))
sns.barplot(data=ver_df, x='version', y='recall_at_5', palette='magma')

plt.title("Real Model Performance per Obfuscation Level")
plt.ylabel("Recall@5 (Version-specific evaluation)")
plt.ylim(0, 1.0)

plt.axhline(
    y=res_df['r_at_5'].mean(),
    color='r',
    linestyle='--',
    label='Overall optimistic average'
)

plt.legend()
plt.savefig("strict_breakdown.png")
plt.show()

### Technical Audit: Why Does the Model Fail

In [ ]:
# Inspect best and worst performing classes
print("="*50)
print("🏆 TOP-10 BEST CLASSES (Model understands them well)")
print("="*50)

best_classes = res_df.sort_values(by=['mrr', 'r_at_1'], ascending=[False, False]).head(10)
print(best_classes[['category', 'behavior', 'r_at_1', 'mrr']])

print("\n" + "="*50)
print("🗑️ TOP-30 WORST CLASSES (Model is effectively blind to them)")
print("="*50)

worst_classes = res_df.sort_values(by=['mrr', 'r_at_1'], ascending=[True, True]).head(30)
print(worst_classes[['category', 'behavior', 'r_at_1', 'mrr']])

# Inspect worst-performing class in detail
print("\n" + "="*50)
print("🔍 ANALYSIS OF WORST ASM CLASS")
print("="*50)

worst_behavior = worst_classes.iloc[0]['behavior']
print(f"Worst-performing class: {worst_behavior}")

# Extract a sample ASM (O2 version) for inspection
sample_asm = df[
    (df['behavior'] == worst_behavior) &
    (df['compiler'] == 'gcc_O2')
]['raw_asm'].iloc[0]

print(f"\nASM code (O2):\n{sample_asm[:500]} ...")

In [ ]:
# Inspect outliers
import pandas as pd

# Add assembly length as a feature (in lines)
df['asm_len'] = df['raw_asm'].astype(str).apply(lambda x: len(x.split('\n')))

print("=== FUNCTIONS WITH SHORTEST ASSEMBLY (Suspicious heavy optimization / stripped code) ===")
print(
    df[['category', 'behavior', 'compiler', 'obfuscation', 'asm_len']]
    .sort_values('asm_len')
    .head(20)
)

print("\n=== INSPECTION OF 'Check_Admin_Privileges' CODE ===")
admin_samples = df[df['behavior'] == 'Check_Admin_Privileges']

for _, row in admin_samples.head(3).iterrows():
    print(f"\n--- {row['compiler']} | {row['obfuscation']} ---")
    print(row['raw_asm'][:300])  # Print prefix of assembly

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Compute ASM length (in lines)
df['asm_len'] = df['raw_asm'].astype(str).apply(lambda x: len(x.split('\n')))

# 2. Count API calls
# Heuristic: look for "call FunctionName" patterns
# (avoids cases like call eax or call 0x4000)
def count_api_calls(asm):
    if not isinstance(asm, str):
        return 0

    calls = re.findall(r'call\s+([A-Za-z_][A-Za-z0-9_]+)', asm)
    external_calls = [c for c in calls if not c.startswith('INSTR')]

    return len(external_calls)

df['api_count'] = df['raw_asm'].apply(count_api_calls)

# 3. ANALYSIS 1: API survivability
print("="*60)
print("🛡️ API SURVIVAL AUDIT BY BEHAVIOR CATEGORY")
print("="*60)

api_stats = df.groupby('behavior').agg(
    total_samples=('api_count', 'count'),
    samples_with_api=('api_count', lambda x: (x > 0).sum()),
    avg_api_per_func=('api_count', 'mean')
)

api_stats['survival_rate'] = api_stats['samples_with_api'] / api_stats['total_samples']

print(api_stats.sort_values('survival_rate'))

# 4. ANALYSIS 2: Compiler impact (Dead Code Elimination)
print("\n" + "="*60)
print("💀 COMPILER OPTIMIZATION IMPACT (DCE Drop)")
print("="*60)

# Compare average function size across compilers
pivot_len = df.pivot_table(
    index='behavior',
    columns='compiler',
    values='asm_len',
    aggfunc='mean'
)

# O2 vs O0 reduction ratio
pivot_len['O2_Survival_Ratio'] = pivot_len['gcc_O2'] / pivot_len['gcc_O0']

print(
    pivot_len[['gcc_O0', 'gcc_O2', 'gcc_Os', 'O2_Survival_Ratio']]
    .sort_values('O2_Survival_Ratio')
)

# 5. Verdict: how many functions are effectively empty?
empty_funcs = df[df['asm_len'] < 10]

print(
    f"\n🚨 Completely EMPTY functions (likely optimized out): "
    f"{len(empty_funcs)} out of {len(df)}"
)

### Experiment 8: Split Evaluation (Easy vs Hard)

In [ ]:
print("\n" + "="*50)
print("⚖️ EXPERIMENT 8: Split Evaluation (Easy vs Hard)")
print("="*50)

# Define masks
# Easy: simple code, low obfuscation, low optimization
mask_easy = (df['obfuscation'].isin(['V1', 'V2'])) & (df['compiler'].isin(['gcc_O0', 'gcc_O2']))

# Hard: dynamic calls, flattening, heavy obfuscation or size optimization
mask_hard = (df['obfuscation'].isin(['V3', 'V5', 'V4'])) | (df['compiler'] == 'gcc_Os')

print(f"Easy samples: {mask_easy.sum()}")
print(f"Hard samples: {mask_hard.sum()}")

def evaluate_subset(mask, name):
    subset_indices = df[mask].index.tolist()
    if not subset_indices:
        return 0

    subset_tensors = asm_tensors[subset_indices]
    subset_df = df.iloc[subset_indices].reset_index(drop=True)

    # Generate queries only for behaviors present in subset
    unique_beh = subset_df['behavior'].unique()
    q_texts = [b.replace('_', ' ') for b in unique_beh]
    q_vecs = encode_text_queries(q_texts)

    # Compute similarity
    sims = torch.matmul(q_vecs, subset_tensors.T).numpy()

    recalls = []

    for i, beh in enumerate(unique_beh):
        # Ground-truth mask inside subset
        target_mask = (subset_df['behavior'] == beh).values

        row_sims = sims[i]
        sorted_idxs = np.argsort(row_sims)[::-1]

        # Rank of first correct match
        first_correct = np.argmax(target_mask[sorted_idxs]) + 1
        recalls.append(1 if first_correct <= 5 else 0)

    return np.mean(recalls)

r5_easy = evaluate_subset(mask_easy, "Easy")
r5_hard = evaluate_subset(mask_hard, "Hard")

print(f"\nRecall@5 [Easy Set]: {r5_easy:.2%} (V1/V2, O0/O2)")
print(f"Recall@5 [Hard Set]: {r5_hard:.2%} (V3/V5, Os)")

plt.figure(figsize=(8, 5))
sns.barplot(
    x=['Easy Set (Simple/Pointers)', 'Hard Set (Dynamic/Flattened)'],
    y=[r5_easy, r5_hard],
    palette='coolwarm'
)

plt.title("Performance Gap: Easy vs Hard Code")
plt.ylabel("Recall@5")
plt.ylim(0, 1.0)

plt.savefig("easy_vs_hard.png")
plt.show()

### Experiment 9: Mean Average Precision (MAP)

In [ ]:
print("\n" + "="*50)
print("📏 EXPERIMENT 3.2: Mean Average Precision (MAP)")
print("="*50)

# Each behavior has ~15 variants (5 versions × 3 compilers)
# MAP penalizes cases where V3/V5 versions appear deep in ranking

map_scores = []

for i, behavior in enumerate(unique_behaviors):
    # Query vector already computed in query_tensors[i]
    sims = sim_matrix_cross[i].numpy()
    sorted_indices = np.argsort(sims)[::-1]

    # Ground truth mask (all samples of this behavior)
    ground_truth = (df['behavior'].values[sorted_indices] == behavior)

    # Relevant ranks
    relevant_ranks = np.where(ground_truth)[0] + 1

    if len(relevant_ranks) == 0:
        map_scores.append(0)
        continue

    # AP computation: Sum(Precision@k) / Total relevant
    precisions = []
    for k, rank in enumerate(relevant_ranks):
        # k+1 = number of relevant items retrieved so far
        # rank = position in ranked list
        precisions.append((k + 1) / rank)

    map_scores.append(np.mean(precisions))

mean_map = np.mean(map_scores)

print(f"Global MAP: {mean_map:.4f}")
print("Interpretation: 1.0 = perfect ranking (all variants on top), <0.3 = only V1/V2 detected reliably")

# Most fragmented classes (where V1 is found but V5 is lost)
df_map = pd.DataFrame({'behavior': unique_behaviors, 'map': map_scores})

print("\n📉 Most fragmented classes (Low MAP):")
print(df_map.sort_values('map').head(10))

### Experiment 10: Query Ensemble (Prompt Engineering)

In [ ]:
print("\n" + "="*50)
print("🗣️ EXPERIMENT 10: Query Ensemble (Prompt Engineering)")
print("="*50)

# Generate 3 prompt variants for each behavior
# 1. Direct (baseline)
prompts_v1 = [b.replace('_', ' ') for b in unique_behaviors]

# 2. "C code that performs ..."
prompts_v2 = [f"C code function that performs {b.replace('_', ' ')}" for b in unique_behaviors]

# 3. "Implementation of ..."
prompts_v3 = [f"implementation of {b.replace('_', ' ')} algorithm" for b in unique_behaviors]

# Encode
emb_v1 = encode_text_queries(prompts_v1)
emb_v2 = encode_text_queries(prompts_v2)
emb_v3 = encode_text_queries(prompts_v3)

# Ensemble (average embeddings)
emb_ensemble = (emb_v1 + emb_v2 + emb_v3) / 3.0
emb_ensemble = F.normalize(emb_ensemble, p=2, dim=1)  # re-normalize

# Evaluate ensemble performance
sim_matrix_ens = torch.matmul(emb_ensemble, asm_tensors.T)
results_ens = []

for i, behavior in enumerate(unique_behaviors):
    sims = sim_matrix_ens[i].numpy()
    sorted_idxs = np.argsort(sims)[::-1]

    correct_mask = (df['behavior'].values[sorted_idxs] == behavior)
    rank = np.argmax(correct_mask) + 1

    results_ens.append(1 if rank <= 5 else 0)

recall_ens = np.mean(results_ens)

print(f"Recall@5 (Single Prompt): {res_df['r_at_5'].mean():.2%}")
print(f"Recall@5 (Ensemble 3x):   {recall_ens:.2%}")

### Experiment 11: Downstream Task: k-NN Classification

In [ ]:
print("\n" + "="*50)
print("🏷️ EXPERIMENT 11: Downstream Task: k-NN Classification")
print("="*50)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

X = asm_tensors.numpy()
y_cat = df['category'].values
y_beh = df['behavior'].values

# Cross-validation to avoid train/test leakage
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cat = np.empty_like(y_cat)

print("Running 5-Fold Cross-Validation for k-NN classifier...")

# k=5 (majority vote among nearest neighbors)
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')

for train_idx, test_idx in skf.split(X, y_cat):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_cat[train_idx], y_cat[test_idx]

    knn.fit(X_train, y_train)
    y_pred_cat[test_idx] = knn.predict(X_test)

print("\n📊 Classification Report (Categories):")
print(classification_report(y_cat, y_pred_cat, zero_division=0))

# Confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_cat, y_pred_cat, normalize='true')

sns.heatmap(
    cm,
    annot=True,
    fmt='.2f',
    xticklabels=np.unique(y_cat),
    yticklabels=np.unique(y_cat),
    cmap='Blues'
)

plt.title("Confusion Matrix (k-NN Classification)")
plt.ylabel("True Category")
plt.xlabel("Predicted Category")

plt.tight_layout()
plt.savefig("knn_confusion_matrix.png")
plt.show()

In [ ]:
print("\n" + "="*50)
print("🪡 EXPERIMENT 5: Needle in a Haystack (HARD MODE)")
print("="*50)

benign_indices = df.index[df['category'] == 'Benign_Utility'].tolist()
haystack = asm_tensors[benign_indices]

# Test V3 and V5 (hard variants)
targets = [
    ("Reverse_TCP_Connect", "V2"),  # Dynamic API usage
    ("Reverse_TCP_Connect", "V3"),  # Dynamic API usage
    ("Reverse_TCP_Connect", "V5"),  # Dynamic API usage

    ("AES_Block_Encrypt", "V2"),    # Flattened control flow
    ("AES_Block_Encrypt", "V3"),    # Flattened control flow
    ("AES_Block_Encrypt", "V5"),    # Flattened control flow

    ("IsDebuggerPresent_Check", "V2"),  # Pointer arithmetic
    ("IsDebuggerPresent_Check", "V3"),  # Pointer arithmetic
    ("IsDebuggerPresent_Check", "V5")   # Pointer arithmetic
]

for beh, obf in targets:
    # Find specific sample
    candidates = df.index[
        (df['behavior'] == beh) &
        (df['obfuscation'] == obf)
    ].tolist()

    if not candidates:
        print(f"⚠️ Sample {beh} ({obf}) not found in dataset!")
        continue

    needle_idx = candidates[0]
    needle_vec = asm_tensors[needle_idx].unsqueeze(0)

    # Build mixed database
    mixed_db = torch.cat([needle_vec, haystack], dim=0)

    # Query
    query = f"function that implements {beh.replace('_', ' ')}"
    q_vec = encode_text_queries([query])

    # Search
    sims = torch.matmul(q_vec, mixed_db.T).numpy()[0]
    rank = np.argsort(sims)[::-1].tolist().index(0) + 1

    print(f"🕵️ Query: {beh} | Variant: {obf}")
    print(f"   Rank: {rank} / {len(benign_indices) + 1}")

    if rank == 1:
        print("   ✅ FOUND IMMEDIATELY")
    elif rank <= 10:
        print("   ⚠️ FOUND within top-10")
    else:
        print("   ❌ LOST in benign noise")

    print("-" * 30)